<a href="https://colab.research.google.com/github/MissSamyuktha/news-finder-langchain-ai-agent/blob/main/NewsFindr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**<font color=blue>NewsFindr:** Personalized, credible, and streamlined news discovery powered by Agentic AI

### Business Context

NewsFindr is redefining news discovery by delivering real-time news updates tailored to user interests. Traditional search methods and generic news feeds often lead to information overload and inefficiencies, making it challenging for users to access relevant and trustworthy content efficiently.

To address this, NewsFindr wants to leverage Agentic AI to build an AI-powered news retrieval agent that ensures accuracy and credibility. By utilizing a structured, multi-step approach, the system will provide secure, fair, and explainable recommendations - enhancing user engagement, optimizing content discovery, and improving access to timely and relevant news.

### Objective

- Provide real-time, personalized news retrieval to help users discover relevant
content effortlessly.

- Ensure accuracy and credibility by sourcing news from trusted platforms and minimizing misinformation.

- Improve user engagement through seamless content discovery, reducing information overload.

- Streamline the news consumption process by eliminating outdated and irrelevant content, providing a refined reading experience.

##**<font color=blue>Installing and Importing Necessary Libraries and Dependencies**

In [ ]:
#Install all the libraries
%pip install -qU langchain-groq

!pip install -q langchain \
                langchainhub==0.1.20 \
                langchain-experimental==0.0.62\
                langchain_huggingface\
                langchain-groq

!pip install -qU duckduckgo-search
!pip uninstall -y duckduckgo_search
!pip install ddgs

!pip install numpy==1.26

In [ ]:
#importing necessary libraries
import json
import os
import pandas as pd
import sqlite3

from langchain.agents import create_sql_agent, initialize_agent # Added initialize_agent
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.agents.agent_types import AgentType
from langchain.sql_database import SQLDatabase
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain import hub
from langchain.agents import load_tools
from langchain.agents import Tool

from pydantic import BaseModel, Field, ValidationError

from ddgs import DDGS
from typing import List, Optional, Dict

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

### Utilising Groq API Key

Initializes Gorq model with different temperature settings.

In [ ]:
from langchain_groq import ChatGroq  # Import Groq LLM

# Get the API key from Colab secrets
from google.colab import userdata
groq_api_key = userdata.get('GROQ_API_KEY') #Complete the code by calling the API key

# High creativity LLM
llm_high = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct", #Call the Groq model
    temperature=0.7, #Complete the code by providing high temperature
    groq_api_key=groq_api_key)

# Low creativity (deterministic) LLM
llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct", #Call the Groq model
    temperature=0.2, #Complete the code by providing low temperature
    groq_api_key=groq_api_key
)

##**<font color=blue>Checking LLMs response**

A low temperature setting ensures consistent and repeatable responses, making the output more deterministic. It is ideal for tasks requiring accuracy, such as fact-based queries, structured data extraction, and rule-based decision-making.

In [ ]:
# Deterministic Model
for i in range(5):
  response = llm.predict("What are top 5 high paying professions in 2026?") #Complete the code by asking a simple question to the llm
  print(response)

A high temperature setting introduces more randomness and variability in responses, making the output creative and diverse. It is useful for tasks like brainstorming, storytelling, content generation, and exploring multiple perspectives.

In [ ]:
# Creative Model
for i in range(5):
  response = llm_high.predict("What are top 5 high paying professions in 2026?") #asking a simple question to the llm
  print(response)

Deterministic model gave more or less same answers but creative model has some differences in responses.

##<font color=blue>**SQL Agent for Data Retrieval and Customer email verification and reterival**

Facilitates efficient data retrieval and modification, enabling seamless customer information management through SQL interactions

<font color=magenta>**Function setup_sql_agent**</FONT>: Initializes an AI-powered SQL agent for querying and updating a database.
It connects to an SQL database, and sets up an agent and check for the presence of the email in the DB, if not found return as "email not found", if email found return with the "name of the customer and area of interest".

In [ ]:
db = SQLDatabase.from_uri("sqlite:////content/customer.db")   # loading the SQLite database

#Initialize the system message
system_message = "You are an SQL assistant. Your role is to efficiently fetch and retrieve user data from the database. If email not found return 'email not found'."
#Initialize the toolkit with customer database and the LLM
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

#Create the SQL agent with the system message
db_agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=False,
    system_message=SystemMessage(system_message)
)

In [ ]:
query= f"Fetch all the unique email_id"

output=db_agent.invoke(query)

output

{'input': 'Fetch all the unique email_id',
 'output': "['alice.6eb33c45-5@gmail.com', 'alice.d56e53b6-1@gmail.com', 'emma.a88fec03-c@gmail.com', 'george.5483cb53-8@gmail.com', 'hannah.4770b814-2@gmail.com', 'ian.203631a0-b@gmail.com', 'ian.aee37571-7@gmail.com', 'julia.b5a14512-c@gmail.com', 'julia.cf4d1edb-9@gmail.com', 'julia.d77d96f3-3@gmail.com']"}

In [ ]:
email="alice.6eb33c45-5@gmail.com"   # email id which is present in the database to check the response of LLm
query= f"Fetch all records with email_id {email}"

output=db_agent.invoke(query)

output

{'input': 'Fetch all records with email_id alice.6eb33c45-5@gmail.com',
 'output': 'The record for \'alice.6eb33c45-5@gmail.com\' is:\n- id: 4\n- customer_id: 6EB33C45-5\n- name: Alice\n- email: alice.6eb33c45-5@gmail.com\n- interests: ["Politics", "Technology", "Business"]\n- last_updated: 2025-03-26 06:46:15'}

In [ ]:
email="sam@gmail.com" # email id which is not present in the database to check the response of LLm

query= f"Fetch all the Interests with email_id {email} in a list"

output=db_agent.invoke(query)

output

{'input': 'Fetch all the Interests with email_id sam@gmail.com in a list',
 'output': '[]'}

As we used deterministic model, model ouputs only data which is present in db and returns none if no results found without hallucinating responses.

##<font color=blue>**Generating Expanded Search Queries for Current News**

Generating Expanded Search Queries for Current News
This function, expand_search_queries, generates precise and up-to-date search queries based on the user's interests. It ensures that each query is crafted to retrieve breaking news, trending topics, or recent developments without relying on specific years.

The function takes user interests and User_query as inputs and processes  through an LLM  to generate a relevant search query.

A system prompt guides the model to focus only on current events and avoid outdated information.

The expanded search queries are stored in expanded_search and returned for later use in retrieving news articles.

In [ ]:
def expand_search_queries(inputs) -> list:
    """
    Expand user interests into time-sensitive search queries for breaking news.
    Accepts either a dict (with 'interests' and 'user_query') or a raw string.
    """
    if isinstance(inputs, dict):
        interests = inputs.get("interests", [])
        user_query = inputs.get("user_query","")
    else:
        interests = [i.strip() for i in str(inputs).split(",")]
        user_query = ""

    system_prompt = """You are a helpful assistant that expands user interests
    into precise, time-sensitive search queries for breaking news.
    Always generate queries that combine the interest with the user query
    and emphasize recency (e.g., 'latest', 'breaking', '2026')."""

    expanded_queries = []
    for interest in interests:
        prompt = f"Generate one search query related to: '{interest}' considering the user query: '{user_query}'"
        response = llm.predict_messages(
            [
                SystemMessage(content=system_prompt),
                HumanMessage(content=prompt)
            ]
        )
        query = response.content.strip()
        if query:
            expanded_queries.append(query)

    return expanded_queries


In [ ]:
# Define the tool
expand_tool = Tool(
    name="ExpandSearchQueries",
    func=expand_search_queries,
    description="Expands user interests into precise, time-sensitive news search queries based on a user query."
)

##<font color=blue>**Fetch News Results Using DuckDuckGo**

This function retrieves news results from DuckDuckGo based on the user's expanded search queries.

In [ ]:
def ddg_search(query: str) -> str:
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=5):   # top 5 results
            title = r.get("title", "")
            url = r.get("href", "")
            body = r.get("body", "")
            res={"title":title,"url":url,"body":body}
            results.append(res)
    return results

In [ ]:
# define the tool
ddg_search_tool = Tool(
    name="DuckDuckGoSearch",
    func=ddg_search,
    description="Searches DuckDuckGo for recent news and returns top 5 results with URLs."
)

##<font color=blue>**Filter Relevant and Trustworthy URLs Based on User Interests**

This function filters search result URLs, assesses the credibility of news sources, and retains only those that are relevant to the user's interests.

Extracts URLs and their body content from the search results.

Updates the state with the refined list of relevant URLs and their descriptions.

In [ ]:
def filter_with_llm(search_results: list) -> list:
    search_results= json.dumps(search_results, indent=2)
    system_prompt = """You are a credibility evaluator.
    Only return search results from trusted, reputable news organizations such as:
    BBC, Reuters, New York Times, CNN, Al Jazeera, The Guardian, Associated Press, Bloomberg.
    Discard results from blogs, company sites, or unknown domains.
    Return only the credible results in JSON format."""


    prompt = "From the following search results, keep only those from the trusted domains listed above. Remove all others:\n" + search_results


    response = llm.predict_messages(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=prompt)
        ]
    )
    return response.content


In [ ]:
# define the tool
credibility_tool = Tool(
    name="CredibilityFilter",
    func=filter_with_llm,
    description="Filters search results by evaluating URL credibility using LLM."
)

##<FONT COLOR=BLUE>**Generate summary for the URLs**

This function generates summaries for the credible URLs

Returns the URLs and the Summary based on the users interest

In [ ]:
def summarize_news(url_list) -> dict:
    """
    Summarizes key news points from a list of URLs and returns both the summary and the URLs.
    """
    system_prompt = """You are a summarization assistant.
    Summarize the key points from the following news URLs in 3–4 sentences.
    Focus on timeliness and relevance."""

    prompt = "Summarize the following news articles:\n" + "\n".join(url_list)

    response = llm.predict_messages(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=prompt)
        ]
    )

    summary_text = response.content
    return {"summary": summary_text, "urls": url_list}


In [ ]:
# define the tool
summarize_tool = Tool(
    name="SummarizeNews",
    func=summarize_news,
    description="Generates summaries from news URLs and returns both the summary and the links."
)

## Tools

In [ ]:
expand_tool = Tool(
    name="ExpandSearchQueries",
    func=expand_search_queries,
    description="Expands user interests into precise, time-sensitive news search queries based on a user query."
)

ddg_search_tool = Tool(
    name="DuckDuckGoSearch",
    func=ddg_search,
    description="Searches DuckDuckGo for recent news and returns top 5 results with URLs."
)

credibility_tool = Tool(
    name="CredibilityFilter",
    func=filter_with_llm,
    description="Filters search results by evaluating URL credibility using LLM."
)

summarize_tool = Tool(
    name="SummarizeNews",
    func=summarize_news,
    description="Generates summaries from news URLs and returns both the summary and the links."
)

##<font color=blue>**Creating an Agent**

In [ ]:
tools = [expand_tool, ddg_search_tool, credibility_tool, summarize_tool]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)


##<font color=blue>**Display Final Verified URLs and Summary based on the User interest and User query**

In [ ]:
email = "alice.6eb33c45-5@gmail.com"   # Example email present in DB
interest_result = db_agent.invoke(f"Fetch all the Interests with email_id {email} in a list")

# Normalize to list
if isinstance(interest_result["output"], dict) and "items" in interest_result["output"]:
    interests = interest_result["output"]["items"]
elif isinstance(interest_result["output"], str):
    interests = [i.strip() for i in interest_result["output"].split(",")]
else:
    interests = interest_result["output"]

user_query = "latest developments in AI regulation"

agent_prompt = f"""
The user's interest is: {interests} and specifically looking for {user_query}.
Follow the process:
1. Expand interests into queries.
2. Search DuckDuckGo.
3. Filter credible results.
4. Summarize top 3.
"""

response = agent.run(agent_prompt)
print("\n======= FINAL RESPONSE =======")
print(response)


To address the user's interest in the latest developments in AI regulation, specifically within the domains of Politics, Technology, and Business, I need to follow the outlined process.

Thought: First, I need to expand the user's interests into precise, time-sensitive news search queries.

Action: ExpandSearchQueries
Action Input: ['["Politics"', '"Technology"', '"Business"]'] and specifically looking for latest developments in AI regulation
Observation: ['It seems like you didn\'t provide a user query. Could you please provide the user query you\'d like me to generate a search query for?\n\nIf you\'d like, I can generate a general search query related to politics. Here it is:\n\n"What are the latest political news updates 2026?"', 'It seems like you forgot to include the user query.\n\nIf you\'d like, I can generate a search query related to "Technology" with a recent focus. Here\'s an example:\n\n"What are the latest advancements in artificial intelligence technology 2026?"\n\nOr, i

Only credible sources filtered out in search results and summary generated from those articles.

##<font color=blue>**Retrieve URLs and Summaries for Three Areas of Interest**

In [ ]:
def query_response(email: str, user_query: str) -> str:
    # Fetch interests based on the provided email
    interest_result = db_agent.invoke(f"Fetch all the Interests with email_id {email} in a list")

    # Normalize to list
    if isinstance(interest_result["output"], dict) and "items" in interest_result["output"]:
      interests = interest_result["output"]["items"]
    elif isinstance(interest_result["output"], str):
        interests = [i.strip() for i in interest_result["output"].split(",")]
    else:
       interests = interest_result["output"]


    # Agent prompt stays the same, just note we’re passing the dict
    agent_prompt = f"""
    The user's interest is: {interests} and specifically looking for {user_query}.
    Here is the process to follow:
    1. Expand the user's interest into news search queries using the 'ExpandSearchQueries' tool.
       The input to this tool should be a dictionary like: {{'interests': {interests}, 'user_query': '{user_query}'}}.
    2. Use the query generated in step 1 to search for recent news using the 'DuckDuckGoSearch' tool for all the queries.
    3. Pass the combined results of all queries into 'CredibilityFilter' for filtering.
    4. From the filtered results, select the top 3 latest news articles for summarization.
    5. Summarize all credible results into a detailed summary using the 'SummarizeNews'.
    6. Provide the final summary to the user.
    """
    response = agent.run(agent_prompt)
    print("\n======= FINAL RESPONSE =======")
    print(response)

    return response

## Checking for responses

In [ ]:
#Check for the user 1
email = "alice.6eb33c45-5@gmail.com"
user_query = "AI in healthcare"
response = query_response(email, user_query)
response

To begin, I need to expand the user's interest into precise, time-sensitive news search queries based on their interests and specific query about AI in healthcare.

Action: ExpandSearchQueries
Action Input: {'interests': ['["Politics"', '"Technology"', '"Business"]'], 'user_query': 'AI in healthcare'}
Observation: ['It seems like your request got cut off. Based on what you\'ve provided, here is a search query that combines an interest in politics with the need for something recent:\n\n"latest political news 2026"\n\nIf you could provide more details or clarify the user query, I\'d be happy to generate a more targeted search query for you!', 'It seems like you forgot to include the user query.\n\nIf you\'d like, I can generate a search query related to "Technology" with a recent focus. Here\'s an example:\n\n"What are the latest advancements in artificial intelligence technology 2026?"\n\nOr, if you provide the user query, I can create a more tailored search query.', 'Since there is no 

'## Summary of AI in Healthcare\n\nThe use of Artificial Intelligence (AI) in healthcare is transforming the industry in various ways. Here are the summaries of the top 3 latest news articles:\n\n### Article 1: Peer Review of “Artificial Intelligence in Healthcare: 2023 Year in Review” - PMC\n\nThe article discusses a significant increase in AI-related healthcare publications in 2023, with a total of 23,306 articles, representing a 133.7% increase. This surge indicates a growing interest and research in AI applications in healthcare. The article likely provides insights into the latest scientific studies and research findings on AI in healthcare.\n\n### Article 2: 7 ways AI is transforming healthcare - The World Economic Forum\n\nAI technologies are being used to improve healthcare outcomes, enhance patient care, and streamline clinical workflows. The article highlights the potential benefits and challenges of AI adoption in the healthcare sector. It features expert insights and exampl

In [ ]:
#Check for the user 2
email = "emma.a88fec03-c@gmail.com"
user_query = "Mobile Technologies"
response = query_response(email, user_query)

In [ ]:
#Check for the user 3
email = "hannah.4770b814-2@gmail.com"
user_query = "climate change policy"
response = query_response(email, user_query)

Only credible sources filtered out in search results and summary generated from those articles.

##<font color=blue>**Conclusion**

The **NewsFindr Agentic AI pipeline** successfully demonstrates how generative AI can transform news discovery into a personalized, credible, and streamlined experience. By integrating:

- **Groq LLMs** for both creative and deterministic reasoning,
- **SQL agents** ensuring personalized retrieval of customer interests,
- **DuckDuckGo search** tools for real-time news updates aligned with those interests,
- **Credibility filters** to ensure trustworthy sources, and reduce misinformation,
- **Summarization agents** to reduce information overload,

The system delivers a refined reading experience tailored to each user's needs.

This multi-step agentic approach ensures:
- **Accuracy & Trustworthiness** by filtering results through reputable domains.
- **Personalization** by linking customer interests directly to search queries.
- **Efficiency** by summarizing complex news into concise insights.
- **Engagement** by reducing noise and focusing on relevant updates.

In essence, NewsFindr provides a blueprint for how **Agentic AI in NLP** can enhance content discovery, minimize misinformation, and empower users with timely, relevant, and explainable news recommendations.

This project highlights the potential of combining structured data retrieval with generative intelligence to redefine the way we consume information in the digital age.

**NewsFindr: Personalized, credible, and streamlined news discovery powered by Agentic AI**